# 05 — Privacy Concerns (Edoardo Balzano)

This notebook focuses on the privacy implications of the student early-warning system. It explores:
1. **Sensitive Feature Handling**: Identifying and masking demographic features.
2. **k-Anonymity**: Assessing re-identification risk in the OULAD dataset.
3. **Privacy-Preserving Explanations**: Ensuring SHAP/LIME explanations do not leak sensitive personal traits.

In [ ]:
import sys
from pathlib import Path
import joblib
import pandas as pd
import numpy as np

# Add src to path
sys.path.append(str(Path.cwd().parent))

from src.privacy import (
    apply_feature_masking, 
    check_k_anonymity, 
    suppress_sensitive_from_explanation,
    SENSITIVE_COLS
)

data_dir = Path.cwd().parent / "data"

# Load shared artifacts
try:
    model   = joblib.load(data_dir / "base_model.pkl")
    X_train = joblib.load(data_dir / "X_train.pkl")
    X_test  = joblib.load(data_dir / "X_test.pkl")
    y_test  = joblib.load(data_dir / "y_test.pkl")
    print("Successfully loaded shared artifacts.")
except FileNotFoundError:
    print("Error: Shared artifacts not found in data/. Please run 02_base_model.ipynb first.")

## 1. k-Anonymity Assessment

We evaluate if the dataset provides enough anonymity for students based on quasi-identifiers.

In [ ]:
# Define quasi-identifiers for k-anonymity check
quasi_ids = ["gender", "region", "highest_education", "age_band"]

# Check k-anonymity on the test set
violations = check_k_anonymity(X_test, quasi_ids, k=5)

## 2. Feature Masking & Utility Trade-off

Compare the model performance with and without sensitive features.

In [ ]:
# Example: Masking sensitive columns
X_test_masked = apply_feature_masking(X_test, drop_sensitive=True)
print(f"Original columns: {len(X_test.columns)}")
print(f"Masked columns: {len(X_test_masked.columns)}")